In [ ]:
import inspect
import json
import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, get_peft_model
from score import score_wer
from torch.utils.data import SequentialSampler
from transformers import AutoModelForCTC, AutoProcessor, Trainer, TrainingArguments

In [ ]:
# Core config (in-notebook)
BASE_MODEL_ID = "nvidia/parakeet-ctc-1.1b"
PROCESSOR_FROM_PRETRAINED_KWARGS = {}
DATASET_ID = "quinnlue/asr-test"
AUDIO_COLUMN = "audio_path"
TEXT_COLUMN = "orthographic_text"
OUTPUT_DIR = Path("artifacts/parakeet-ctc-lora")
SAMPLE_RATE = 16000

# TrainingArguments are defined in-notebook (not in config.json)
TRAINING_ARGS_CFG = {
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "learning_rate": 1e-5,
    "weight_decay": 0.005,
    "warmup_steps": 100,
    "warmup_ratio": 0.0,
    "num_train_epochs": 3.0,
    "max_steps": -1,
    "max_grad_norm": 1.0,
    "lr_scheduler_type": "linear",
    "optim": "adamw_torch",
    "adam_beta1": 0.9,
    "adam_beta2": 0.98,
    "adam_epsilon": 1e-8,
    "bf16": True,
    "fp16": False,
    "tf32": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_use_reentrant": False,
    "evaluation_strategy": "epoch",
    "eval_steps": None,
    "save_strategy": "epoch",
    "save_steps": None,
    "save_total_limit": 2,
    "save_safetensors": True,
    "logging_strategy": "steps",
    "logging_steps": 10,
    "logging_first_step": False,
    "report_to": ["wandb"],
    "run_name": None,
    "load_best_model_at_end": False,
    "metric_for_best_model": "wer",
    "greater_is_better": False,
    "remove_unused_columns": False,
    "dataloader_pin_memory": True,
    "dataloader_num_workers": 0,
    "group_by_length": False,
    "length_column_name": "input_length",
    "seed": 42,
    "data_seed": None,
}

# LoRA params
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_BIAS = "none"
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Optional smoke test
SMOKE_TEST = False
SMOKE_TRAIN_EXAMPLES = 64
SMOKE_EVAL_EXAMPLES = 16

# Runtime
DEVICE = torch.device("cuda")
DTYPE = torch.bfloat16

# Weights & Biases
WANDB_CFG = {
    "project": "parakeet-ctc-lora",
    "log_model": "false",
    "api_key_env_var": "WANDB_API_KEY",
}
WANDB_API_KEY_ENV_VAR = WANDB_CFG["api_key_env_var"]
if WANDB_API_KEY_ENV_VAR and os.getenv(WANDB_API_KEY_ENV_VAR):
    os.environ["WANDB_API_KEY"] = os.getenv(WANDB_API_KEY_ENV_VAR)

# Augmentation config (kept in notebook for future use)
AUGMENTATION_CFG = {
    "waveform": {
        "time_stretch_prob": 0.3,
        "rir_prob": 0.5,
        "noise_prob": 0.5,
        "min_speed": 0.9,
        "max_speed": 1.1,
        "max_stretch_samples": 480000,
        "min_snr_db": 5.0,
        "max_snr_db": 30.0,
    },
    "vtlp": {
        "prob": 0.3,
        "alpha_min": 0.9,
        "alpha_max": 1.1,
        "fhi": 4800.0,
    },
    "spec": {
        "prob": 1.0,
        "policy": "LB",
    },
    "noise_dataset": None,
    "rir_dataset": None,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if WANDB_CFG.get("project") is not None:
    os.environ["WANDB_PROJECT"] = str(WANDB_CFG["project"])
if WANDB_CFG.get("log_model") is not None:
    os.environ["WANDB_LOG_MODEL"] = str(WANDB_CFG["log_model"])

In [5]:
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, **PROCESSOR_FROM_PRETRAINED_KWARGS)
model = AutoModelForCTC.from_pretrained(
    BASE_MODEL_ID,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
)

ds = load_dataset(DATASET_ID)
train_dataset = ds["train"]
eval_dataset = ds["validation"]
test_dataset = ds["test"]

if SMOKE_TEST:
    train_dataset = train_dataset.select(range(SMOKE_TRAIN_EXAMPLES))
    eval_dataset = eval_dataset.select(range(SMOKE_EVAL_EXAMPLES))

for name, split in [("train", train_dataset), ("eval", eval_dataset), ("test", test_dataset)]:
    print(name, len(split))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/1694 [00:00<?, ?it/s]

train 1024
eval 256
test 256


In [ ]:
@dataclass
class CTCDataCollator:
    processor: AutoProcessor
    audio_column: str
    text_column: str
    sampling_rate: int
    dtype: torch.dtype
    pad_token_id: int = 1024  # model's pad_token_id Ã¢â‚¬â€ used to mask label padding
    subsampling_factor: int = 4  # FastConformer conv subsampling ratio

    def __call__(self, batch):
        audio = [item[self.audio_column]["array"] for item in batch]
        text = [item[self.text_column] for item in batch]

        inputs = self.processor(
            audio=audio,
            sampling_rate=self.sampling_rate,
            return_tensors="pt",
            padding=True,
        )

        labels = self.processor.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
        )

        label_ids = labels["input_ids"]
        # Ã¢Å¡Â  Must use the model's pad_token_id (1024), NOT -100.
        # The model computes target_lengths as (labels != pad_token_id).sum(),
        # so using -100 causes it to count padding tokens as real labels,
        # violating the CTC length constraint and causing CUDA illegal memory access.
        label_ids = label_ids.masked_fill(labels["attention_mask"].ne(1), self.pad_token_id)

        # ---- CTC length guard ------------------------------------------------
        # CTC loss requires: input_length >= target_length for every sample.
        # The encoder downsamples time by `subsampling_factor`, so the logit
        # sequence length Ã¢â€°Ë† n_frames // subsampling_factor.
        # If any sample violates this, skip it to avoid a CUDA illegal-memory-access.
        n_frames = inputs["attention_mask"].sum(dim=-1)  # (B,)
        logit_lengths = n_frames // self.subsampling_factor
        target_lengths = (label_ids != self.pad_token_id).sum(dim=-1)  # (B,)

        keep = logit_lengths >= target_lengths
        if not keep.all():
            keep_idx = keep.nonzero(as_tuple=True)[0]
            if len(keep_idx) == 0:
                raise ValueError(
                    "Every sample in this batch violates the CTC length "
                    "constraint (label longer than downsampled input). "
                    "Consider filtering short audio from the dataset."
                )
            inputs["input_features"] = inputs["input_features"][keep_idx]
            inputs["attention_mask"] = inputs["attention_mask"][keep_idx]
            label_ids = label_ids[keep_idx]
        # ----------------------------------------------------------------------

        # Let Trainer handle device placement Ã¢â‚¬â€ return CPU tensors
        return {
            "input_features": inputs["input_features"].to(dtype=self.dtype),
            "attention_mask": inputs["attention_mask"],
            "labels": label_ids,
        }

In [8]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias=LORA_BIAS,
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()
model = model.to(DEVICE)

trainable params: 5,505,024 || all params: 1,068,045,313 || trainable%: 0.5154


In [ ]:
data_collator = CTCDataCollator(
    processor=processor,
    audio_column=AUDIO_COLUMN,
    text_column=TEXT_COLUMN,
    sampling_rate=SAMPLE_RATE,
    dtype=DTYPE,
)

EVAL_PREDICTIONS_PATH = OUTPUT_DIR / "eval_predictions.jsonl"

def _json_safe_value(value):
    if isinstance(value, np.generic):
        return value.item()
    return value

def _build_eval_export_rows(eval_ds, pred_texts, eval_step):
    references = []
    rows = []
    for idx in range(len(eval_ds)):
        example = eval_ds[idx]
        row = {
            key: _json_safe_value(value)
            for key, value in example.items()
            if key != AUDIO_COLUMN
        }
        reference_text = str(example.get(TEXT_COLUMN, ""))
        predicted_text = str(pred_texts[idx])
        row["eval_step"] = int(eval_step)
        row["reference_text"] = reference_text
        row["predicted_text"] = predicted_text
        row["example_wer"] = float(score_wer(actual=[reference_text], predicted=[predicted_text]))
        references.append(reference_text)
        rows.append(row)
    corpus_wer = float(score_wer(actual=references, predicted=pred_texts))
    return rows, corpus_wer

def _append_jsonl(path, rows):
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\\n")

def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    pred_ids = np.argmax(logits, axis=-1)

    label_ids = eval_pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.tokenizer.batch_decode(label_ids, group_tokens=False)

    corpus_wer = float(score_wer(actual=label_str, predicted=pred_str))
    return {"wer": corpus_wer, "corpus_wer": corpus_wer}

TRAINING_ARG_ALIASES = {
    "eval_strategy": "evaluation_strategy",
}
valid_training_arg_keys = set(inspect.signature(TrainingArguments.__init__).parameters)
training_args_kwargs = {}
for raw_key, value in TRAINING_ARGS_CFG.items():
    normalized_key = TRAINING_ARG_ALIASES.get(raw_key, raw_key)
    if normalized_key in valid_training_arg_keys:
        training_args_kwargs[normalized_key] = value

training_args_kwargs["output_dir"] = str(OUTPUT_DIR)
training_args = TrainingArguments(**training_args_kwargs)

class NoShuffleTrainer(Trainer):
    def _get_train_sampler(self, dataset=None):
        return SequentialSampler(dataset if dataset is not None else self.train_dataset)

class ExportingNoShuffleTrainer(NoShuffleTrainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        metrics = super().evaluate(eval_dataset=eval_dataset, ignore_keys=ignore_keys, metric_key_prefix=metric_key_prefix)
        target_eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        if target_eval_dataset is None or len(target_eval_dataset) == 0:
            return metrics
        predictions = self.predict(target_eval_dataset, metric_key_prefix="export_eval").predictions
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        pred_ids = np.argmax(predictions, axis=-1)
        pred_texts = processor.batch_decode(pred_ids)
        rows, corpus_wer = _build_eval_export_rows(target_eval_dataset, pred_texts, self.state.global_step)
        _append_jsonl(EVAL_PREDICTIONS_PATH, rows)
        metrics[f"{metric_key_prefix}_corpus_wer_export"] = corpus_wer
        return metrics

trainer = ExportingNoShuffleTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [11]:
# Smoke-check one collated batch before launching Trainer.
sample_batch = [train_dataset[i] for i in range(2)]
sample_inputs = data_collator(sample_batch)
sample_inputs = {k: v.to(DEVICE) for k, v in sample_inputs.items()}

model.eval()
with torch.no_grad():
    sample_outputs = model(**sample_inputs)

print({
    "loss": float(sample_outputs.loss),
    "input_shape": tuple(sample_inputs["input_features"].shape),
    "label_shape": tuple(sample_inputs["labels"].shape),
})

{'loss': 3.105595588684082, 'input_shape': (2, 1274, 80), 'label_shape': (2, 41)}


In [ ]:
# ── Diagnostic cell 1: Check what the model actually sees during training ──
# Compare eval-mode (smoke test) vs train-mode loss on the SAME batch
import torch

sample_batch = [train_dataset[i] for i in range(2)]
sample_inputs = data_collator(sample_batch)
sample_inputs_gpu = {k: v.to(DEVICE) for k, v in sample_inputs.items()}

# Eval mode (same as smoke test)
model.eval()
with torch.no_grad():
    eval_out = model(**sample_inputs_gpu)
print(f"eval-mode loss: {eval_out.loss.item():.4f}")

# Train mode (same as Trainer)
model.train()
with torch.no_grad():
    train_out = model(**sample_inputs_gpu)
print(f"train-mode loss: {train_out.loss.item():.4f}")

# If these are very different, dropout is contributing (but shouldn't be 10x)
print(f"ratio: {train_out.loss.item() / eval_out.loss.item():.2f}x")

In [ ]:
# ── Diagnostic cell 2: Test with a FULL training-sized batch ──
# The smoke test only uses 2 samples; Trainer uses batch_size=16.
# Larger batches have more padding → check if CTC loss scales badly.
import torch

batch_size = min(16, len(train_dataset))
full_batch = [train_dataset[i] for i in range(batch_size)]
full_inputs = data_collator(full_batch)
full_inputs_gpu = {k: v.to(DEVICE) for k, v in full_inputs.items()}

model.eval()
with torch.no_grad():
    full_out = model(**full_inputs_gpu)

print(f"batch_size={batch_size}")
print(f"input_features shape: {full_inputs['input_features'].shape}")
print(f"labels shape:         {full_inputs['labels'].shape}")
print(f"attention_mask shape: {full_inputs['attention_mask'].shape}")
print(f"loss: {full_out.loss.item():.4f}")
print(f"logits shape: {full_out.logits.shape}")

# Check attention mask stats
attn = full_inputs["attention_mask"]
for i in range(batch_size):
    n_real = attn[i].sum().item()
    n_total = attn[i].shape[0]
    pad_pct = 100 * (1 - n_real / n_total)
    label_len = (full_inputs["labels"][i] != 1024).sum().item()
    print(f"  sample {i}: {n_real}/{n_total} real frames ({pad_pct:.1f}% pad), label_len={label_len}")

In [ ]:
# ── Diagnostic cell 3: Inspect how the model computes CTC loss internally ──
# Check pad_token_id, blank_id, and whether the model uses attention_mask
# to compute input_lengths (critical for CTC).
import torch

print("=== Model config ===")
base_model = model.base_model.model if hasattr(model, 'base_model') else model
cfg = base_model.config
print(f"pad_token_id:  {cfg.pad_token_id}")
print(f"vocab_size:    {cfg.vocab_size}")
print(f"ctc_loss_reduction: {getattr(cfg, 'ctc_loss_reduction', 'NOT SET')}")
print(f"ctc_zero_infinity:  {getattr(cfg, 'ctc_zero_infinity', 'NOT SET')}")
print(f"blank token id: {getattr(cfg, 'blank_token_id', getattr(cfg, 'blank_id', 'NOT SET'))}")

print("\n=== Processor/tokenizer ===")
print(f"tokenizer.pad_token_id: {processor.tokenizer.pad_token_id}")
print(f"tokenizer.vocab_size:   {processor.tokenizer.vocab_size}")

# Check if model forward actually uses attention_mask for input_lengths
print("\n=== Forward signature ===")
import inspect
fwd = base_model.forward
sig = inspect.signature(fwd)
print(f"forward params: {list(sig.parameters.keys())}")

# Manually compute CTC loss to compare
print("\n=== Manual CTC loss vs model loss ===")
sample_batch = [train_dataset[i] for i in range(4)]
sample_inputs = data_collator(sample_batch)
sample_inputs_gpu = {k: v.to(DEVICE) for k, v in sample_inputs.items()}

model.eval()
with torch.no_grad():
    out = model(**sample_inputs_gpu)
    logits = out.logits  # (B, T, V)
    
    # Compute input_lengths the way the model likely does
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1).transpose(0, 1)  # (T, B, V)
    
    labels = sample_inputs_gpu["labels"]
    attn_mask = sample_inputs_gpu["attention_mask"]
    
    # Method 1: from attention_mask (correct)
    # The model's encoder downsamples - logits time dim tells us the actual output length
    input_lengths_from_logits = torch.full((logits.shape[0],), logits.shape[1], dtype=torch.long, device=DEVICE)
    
    # Method 2: check if model uses attention_mask
    # For models that respect attention_mask, input_lengths should be shorter for padded samples
    
    target_lengths = (labels != cfg.pad_token_id).sum(dim=-1)
    
    print(f"logits shape: {logits.shape}")
    print(f"input_lengths (from logit dim): {input_lengths_from_logits.tolist()}")
    print(f"target_lengths: {target_lengths.tolist()}")
    print(f"model loss: {out.loss.item():.4f}")
    
    # Manual CTC loss with same params
    manual_loss = torch.nn.functional.ctc_loss(
        log_probs, labels, input_lengths_from_logits, target_lengths,
        blank=cfg.pad_token_id, reduction="mean", zero_infinity=False,
    )
    print(f"manual CTC loss (all logit frames): {manual_loss.item():.4f}")

In [ ]:
# ── Diagnostic cell 4: Per-sample loss across multiple random batches ──
# Check if a few outlier samples are driving the high average loss.
import torch, random

model.eval()
per_sample_losses = []

# Test 5 random batches of 16
for batch_idx in range(5):
    indices = random.sample(range(len(train_dataset)), min(16, len(train_dataset)))
    batch = [train_dataset[i] for i in indices]
    inputs = data_collator(batch)
    inputs_gpu = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = model(**inputs_gpu)
        logits = out.logits
        
        # Compute per-sample CTC loss
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1).transpose(0, 1)
        labels = inputs_gpu["labels"]
        cfg = (model.base_model.model if hasattr(model, 'base_model') else model).config
        
        input_lengths = torch.full((logits.shape[0],), logits.shape[1], dtype=torch.long, device=DEVICE)
        target_lengths = (labels != cfg.pad_token_id).sum(dim=-1)
        
        for i in range(logits.shape[0]):
            loss_i = torch.nn.functional.ctc_loss(
                log_probs[:, i:i+1, :], 
                labels[i:i+1], 
                input_lengths[i:i+1], 
                target_lengths[i:i+1],
                blank=cfg.pad_token_id, 
                reduction="mean", 
                zero_infinity=False,
            )
            per_sample_losses.append((indices[i] if i < len(indices) else -1, loss_i.item()))
    
    print(f"batch {batch_idx}: model_loss={out.loss.item():.4f}")

# Show distribution
losses = [l for _, l in per_sample_losses]
print(f"\n=== Per-sample loss stats (n={len(losses)}) ===")
print(f"mean={np.mean(losses):.4f}, median={np.median(losses):.4f}")
print(f"min={np.min(losses):.4f}, max={np.max(losses):.4f}")
print(f"std={np.std(losses):.4f}")
print(f"samples with loss > 10: {sum(1 for l in losses if l > 10)}/{len(losses)}")
print(f"samples with loss > 20: {sum(1 for l in losses if l > 20)}/{len(losses)}")

# Show top-10 worst
worst = sorted(per_sample_losses, key=lambda x: x[1], reverse=True)[:10]
print(f"\nTop-10 worst samples:")
for idx, loss in worst:
    print(f"  dataset_idx={idx}, loss={loss:.4f}")

In [ ]:
# ── Diagnostic cell 5: Check the model's internal forward for CTC loss ──
# Read the actual source code of the model's forward method to see
# how it computes input_lengths and CTC loss.
import inspect

base_model = model.base_model.model if hasattr(model, 'base_model') else model
print(f"Model class: {type(base_model).__name__}")
print(f"Module: {type(base_model).__module__}")

# Print the forward method source
try:
    src = inspect.getsource(type(base_model).forward)
    # Just print the CTC loss portion (last ~40 lines)
    lines = src.split('\n')
    # Find where loss computation starts
    loss_start = None
    for i, line in enumerate(lines):
        if 'ctc_loss' in line.lower() or 'loss' in line.lower() and 'labels' in line.lower():
            loss_start = max(0, i - 5)
            break
    if loss_start is not None:
        print(f"\n=== Loss computation (lines {loss_start}-{len(lines)}) ===")
        print('\n'.join(lines[loss_start:]))
    else:
        # Print last 50 lines
        print(f"\n=== Last 50 lines of forward() ===")
        print('\n'.join(lines[-50:]))
except Exception as e:
    print(f"Could not get source: {e}")
    # Fallback: check if model has a specific loss method
    print(f"\nModel attributes: {[a for a in dir(base_model) if 'loss' in a.lower() or 'ctc' in a.lower()]}")

In [ ]:
# ── Diagnostic cell 6: Check if Trainer is modifying labels ──
# The Trainer may be replacing pad_token_id with -100 via label_pad_token_id,
# which would BREAK CTC loss computation.
# Also check gradient flow on a single training step.
import torch

print("=== Trainer label handling ===")
print(f"training_args.label_pad_token_id: {getattr(training_args, 'label_pad_token_id', 'NOT SET')}")

# Simulate what Trainer does: call data_collator, then forward
batch = [train_dataset[i] for i in range(4)]
collated = data_collator(batch)

print(f"\nLabels from collator:")
print(f"  unique values: {torch.unique(collated['labels']).tolist()}")
print(f"  min={collated['labels'].min().item()}, max={collated['labels'].max().item()}")
print(f"  contains -100: {(collated['labels'] == -100).any().item()}")
print(f"  contains 1024: {(collated['labels'] == 1024).any().item()}")

# Now do a REAL training step (with backward) and check grad norms
model.train()
inputs_gpu = {k: v.to(DEVICE) for k, v in collated.items()}
out = model(**inputs_gpu)
loss = out.loss
print(f"\nSingle-step train loss: {loss.item():.4f}")

loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
print(f"Total grad norm: {total_norm.item():.4f}")

# Per-module grad norms (top 10 largest)
grad_norms = []
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norms.append((name, param.grad.norm().item()))
grad_norms.sort(key=lambda x: x[1], reverse=True)
print(f"\nTop 10 gradient norms:")
for name, norm in grad_norms[:10]:
    print(f"  {norm:.4f}  {name}")

model.zero_grad()